# UK Labour Housing Promise Tracker

This notebook documents the workflow of our semi-automatic promise tracker website.

The project tracks the implementation progress of Labour's 2024 housing promises by combining:

1. a structured promise dataset;
2. automatically collected evidence from official sources;
3. rule-based automatic status suggestions;
4. an interactive website dashboard.

The tracker is not a fully automated fact-checking system. Instead, it is a semi-automatic prototype: the system collects and suggests evidence, while final political judgement still requires human review.

## Public Website Link

The public dashboard is available here:

(https://uk-labour-promise-tracker-muclrffhxxvvljjzezpvym.streamlit.app/)

## Research Question

To what extent can UK Labour's 2024 manifesto promises on housing be tracked through parliamentary, governmental, and official policy evidence?

## Project Aim

The aim of this project is to explore how political campaign promises can be tracked after an election.

Instead of only asking whether a promise is true or false, this tracker focuses on implementation progress. Each promise is assigned a status and progress score based on available evidence.

In [ ]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR / "data"
SCRIPTS_DIR = BASE_DIR / "scripts"
WEBSITE_DIR = BASE_DIR / "website"

PROMISES_FILE = DATA_DIR / "promises.csv"
EVIDENCE_FILE = DATA_DIR / "evidence.csv"
SUGGESTIONS_FILE = DATA_DIR / "promise_status_suggestions.csv"

print("Project folder:", BASE_DIR)
print("Promises file exists:", PROMISES_FILE.exists())
print("Evidence file exists:", EVIDENCE_FILE.exists())
print("Suggestions file exists:", SUGGESTIONS_FILE.exists())

## Project Structure

The project is organised as follows:

```text
promise-tracker-website/
├── data/
│   ├── promises.csv
│   ├── evidence.csv
│   └── promise_status_suggestions.csv
│
├── scripts/
│   ├── collect_evidence.py
│   ├── update_status.py
│   └── update_tracker.py
│
├── website/
│   └── app.py
│
├── README.md
├── requirements.txt
└── promise_tracker_workflow.ipynb

In [ ]:
```python
promises = pd.read_csv(PROMISES_FILE)

print("Number of promises:", len(promises))
promises.head()

## Promise Dataset

The main promise dataset is stored in `data/promises.csv`.

Each row represents one housing promise. The dataset includes:

- promise ID;
- simplified promise text;
- topic;
- search keywords;
- current human-reviewed status;
- progress score;
- evidence summary;
- parliamentary evidence information.

The keywords are important because they allow the automatic evidence collection script to search official sources for relevant updates.

In [ ]:
promises[[
    "promise_id",
    "promise_text",
    "topic",
    "status",
    "progress_score",
    "parliamentary_status"
]]

In [ ]:
status_counts = promises["status"].value_counts().reset_index()
status_counts.columns = ["status", "count"]

status_counts

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.bar(status_counts["status"], status_counts["count"])
plt.title("Promise Status Distribution")
plt.xlabel("Status")
plt.ylabel("Number of Promises")
plt.xticks(rotation=30)
plt.show()

## Evidence Collection

The tracker uses `scripts/collect_evidence.py` to collect evidence automatically.

The script reads the keywords for each promise and searches GOV.UK for related official documents, policy pages, and government updates.

The collected evidence is saved into `data/evidence.csv`.

Each evidence item includes:

- evidence ID;
- promise ID;
- source type;
- title;
- URL;
- publication date;
- evidence text;
- suggested status;
- collection date.

In [ ]:
if EVIDENCE_FILE.exists():
    evidence = pd.read_csv(EVIDENCE_FILE)
    print("Number of evidence items:", len(evidence))
    display(evidence.head())
else:
    print("No evidence file found. Run scripts/collect_evidence.py first.")

In [ ]:
if EVIDENCE_FILE.exists():
    evidence_count_by_promise = (
        evidence.groupby("promise_id")
        .size()
        .reset_index(name="evidence_count")
        .sort_values("evidence_count", ascending=False)
    )

    evidence_count_by_promise.head(10)

In [ ]:
if EVIDENCE_FILE.exists():
    merged = promises.merge(
        evidence_count_by_promise,
        on="promise_id",
        how="left"
    )

    merged["evidence_count"] = merged["evidence_count"].fillna(0).astype(int)

    merged[[
        "promise_id",
        "topic",
        "status",
        "progress_score",
        "evidence_count"
    ]]

## Automatic Status Suggestions

The tracker uses `scripts/update_status.py` to generate automatic status suggestions.

This script checks the collected evidence and applies simple rule-based logic.

For example:

- if no evidence is found, the suggested status is `not started`;
- if evidence includes words such as `consultation`, `proposal`, `strategy`, or `funding`, the promise may be suggested as `in progress`;
- if evidence includes terms such as `act`, `law`, `legislation`, or `enacted`, the promise may be suggested as `implemented` or `partly implemented`.

These automatic suggestions do not replace human judgement. They are used to support human review.

In [ ]:
if SUGGESTIONS_FILE.exists():
    suggestions = pd.read_csv(SUGGESTIONS_FILE)
    print("Number of automatic suggestions:", len(suggestions))
    display(suggestions.head())
else:
    print("No suggestions file found. Run scripts/update_status.py first.")

In [ ]:
if SUGGESTIONS_FILE.exists():
    suggestions[[
        "promise_id",
        "current_status",
        "suggested_status",
        "current_progress_score",
        "suggested_progress_score",
        "evidence_count"
    ]]

## Comparison Between Human-Reviewed and Auto-Suggested Status

The website displays both the current human-reviewed status and the automatic suggested status.

This makes the tracker more transparent because users can see where the system agrees or disagrees with the current classification.

This is important because political promise tracking cannot be fully automated. Some evidence may be ambiguous, incomplete, or only partially related to the promise.

In [ ]:
if SUGGESTIONS_FILE.exists():
    comparison = suggestions[[
        "promise_id",
        "current_status",
        "suggested_status",
        "evidence_count",
        "auto_summary"
    ]]

    comparison

## Update Pipeline

The full update process is handled by `scripts/update_tracker.py`.

This script runs:

1. `collect_evidence.py`
2. `update_status.py`

This means the tracker can update evidence and regenerate automatic status suggestions with one command.

```bash
python scripts/update_tracker.py

After updating the data, the website can be opened with:

python -m streamlit run website/app.py

In [ ]:
```python
print("To update the tracker, run this command in the terminal:")
print("python scripts/update_tracker.py")

print("\nTo open the website dashboard, run:")
print("python -m streamlit run website/app.py")

## Dashboard

The website dashboard is built with Streamlit.

It includes:

- overall promise progress;
- status distribution chart;
- progress chart for each promise;
- search bar;
- topic and status filters;
- promise detail view;
- collected evidence;
- auto-suggested status;
- update information.

The dashboard is interactive and allows users to explore how each promise is currently classified and what evidence has been collected.

## Methodology

This project uses a semi-automatic method.

First, the promise dataset is manually structured. Each promise is assigned a topic and a set of search keywords. These keywords make it possible for the system to search for relevant official evidence.

Second, the evidence collection script searches GOV.UK for related documents. The results are stored in `evidence.csv`.

Third, the status update script checks the collected evidence and produces an automatic suggested status. The script uses simple keyword-based rules to identify whether the evidence suggests implementation, policy development, or uncertainty.

Finally, the website dashboard displays both the human-reviewed status and the automatic suggested status. This design keeps the tracker transparent. The system can support evidence collection and classification, but final political judgement still requires human review.

## Limitations

This prototype does not fully solve political promise tracking.

There are several limitations:

- GOV.UK search results may include irrelevant documents.
- Some promises require statistical evidence from ONS or budget data, not only GOV.UK pages.
- The automatic status suggestion is rule-based and may be inaccurate.
- Final promise classification still needs human review.
- Some promises may be partly implemented in practice before clear legislation appears.

Because of these limitations, this tracker should be understood as a semi-automatic research prototype, not a fully automated fact-checking system.

## Future Improvements

Future versions could include:

- UK Parliament Bills API integration;
- ONS housing statistics integration;
- budget and spending document tracking;
- better relevance scoring;
- AI-assisted evidence summarisation;
- manual review interface;
- deployment as a public website.

These improvements would make the tracker more accurate and more useful for long-term monitoring.

## Conclusion

This notebook documents the development of a working local prototype for a semi-automatic political promise tracker.

The project shows how structured data, official evidence collection, automatic status suggestions, and an interactive dashboard can be combined to track the implementation of campaign promises.

The tracker is not intended to replace human judgement. Instead, it helps organise evidence and makes the promise-tracking process more transparent, systematic, and easier to update.